# 01 · Build the Gulf of Mexico regional MOM6 case (CrocoDash)

**Goal:** whole-Gulf MOM6 parent for one-way nesting into SFINCS (Galveston Bay first).

**Events:** set `EVENT` in the settings cell and run the notebook once per event. Each event gets its own CESM case.
- `ike`: Hurricane Ike, Sep 2008, landfall at Galveston → main Galveston comparison.
- `katrina`: Hurricane Katrina, Aug 2005, landfall LA/MS → far-field test at Galveston.

All inputs are stock for 2005 and 2008: JRA55-do atmosphere, GLORYS ocean (RDA), TPXO9 tides.
JRA (~55 km, 3-hourly) under-resolves hurricane cores, so peak surge will be low. Holland-vortex winds come later.

**Where to run:** NCAR JupyterHub on **Casper**, kernel **CrocoDash** (installed by `CROCODILEworkspace/install.sh --workshop`).
Request **≥ 40 GB memory, 4 h**. `process_forcings()` regrids GLORYS for the whole Gulf.
Build and submit happen afterwards in a **Derecho** terminal (Section 5).

**Status:** written from the CrocoDash 1.0 API and the CrocoGallery tutorials (`tutorial-ocn`, `dart/tutorial3`). **Not yet run.** Expect small fixes on first use.

Design notes and evidence: `INVESTIGATION_REPORT.md` (§4, §8, §3 datum).

## 0. Settings — edit this cell only

In [ ]:
import os
from pathlib import Path

# --- event: run this notebook once per event -> one CESM case each ---
EVENT = "ike"             # "katrina" or "ike"
EVENTS = {
    # 10+ days spin-up before the storm enters the Gulf
    "katrina": ["2005-08-15 00:00:00", "2005-09-05 00:00:00"],  # landfall 29 Aug 2005 (LA/MS)
    "ike":     ["2008-08-30 00:00:00", "2008-09-20 00:00:00"],  # landfall 13 Sep 2008 (Galveston)
}
DATE_RANGE = EVENTS[EVENT]
CASENAME   = f"gom12_{EVENT}.001"
GRID_NAME  = f"gom12_{EVENT}"   # unique per case: CESM registers grid names globally

# --- paths (match your CROCODILEworkspace install) ---
WORKSPACE = Path("/glade/work/faezehmaghso/crocodile2026")
CESMROOT  = WORKSPACE / "CESM"                      # CESM checkout made by install.sh
INPUTDIR  = WORKSPACE / "croc_input" / CASENAME
CASEROOT  = WORKSPACE / "croc_cases" / CASENAME

PROJECT   = "UCGD0009"   # workshop project; replace with your own NCAR project code

# --- data on GLADE (from CrocoGallery/crocogallery/known_paths.json) ---
GEBCO  = Path("/glade/campaign/cgd/oce/projects/CROCODILE/workshops/2026/CrocoDash/data/gebco_2026/GEBCO_2026_coarse_x8.nc")
TPXO_H = "/glade/campaign/cesm/cesmdata/inputdata/ocn/mom/croc/crocogallerydata/tpxo/h_tpxo9.v1.nc"
TPXO_U = "/glade/campaign/cesm/cesmdata/inputdata/ocn/mom/croc/crocogallerydata/tpxo/u_tpxo9.v1.nc"

# --- domain: whole Gulf of Mexico + NW Caribbean ---
# Longitudes in [0, 360]:  262 = 98W,  282 = 78W ... east edge at 82W cuts the Florida Straits.
RES    = 1/12      # deg (~9 km)
XSTART = 262.0     # 98W
LENX   = 16.0      # -> 82W
YSTART = 18.0
LENY   = 13.0      # -> 31N (north edge on land)
MIN_DEPTH = 5.0    # m; fall back to 9.5 (tutorial value) if the run is unstable
NK = 50            # vertical levels

# --- tides ---
TIDAL_CONSTITUENTS = ["M2", "S2", "N2", "K2", "K1", "O1", "P1", "Q1"]

# --- compute ---
NTASKS_OCN = 128           # one Derecho CPU node
JOB_QUEUE  = "main"
WALLCLOCK  = "02:00:00"

assert GEBCO.exists(), f"GEBCO not found: {GEBCO}"
assert CESMROOT.exists(), f"CESM not found: {CESMROOT}"

## 1. Horizontal grid

In [ ]:
from CrocoDash.grid import Grid

grid = Grid(
    resolution=RES,
    xstart=XSTART,
    lenx=LENX,
    ystart=YSTART,
    leny=LENY,
    name=GRID_NAME,
)
print(f"nx={grid.nx}, ny={grid.ny}, cells={grid.nx * grid.ny:,}")

## 2. Topography

GEBCO 2026 coarsened ×8 (~2 arc-min), enough for 1/12°. `keep_largest_basin()` removes lakes and isolated ponds.

In [ ]:
from CrocoDash.topo import Topo

topo = Topo(grid=grid, min_depth=MIN_DEPTH)
topo.set_from_dataset(
    bathymetry_path=GEBCO,
    longitude_coordinate_name="lon",
    latitude_coordinate_name="lat",
    vertical_coordinate_name="elevation",
    depth_method="xesmf",
    mask_method="dataset",
)
topo.keep_largest_basin()
topo.basintmask.plot(size=5, aspect=grid.nx / grid.ny)
print(f"max depth = {float(topo.max_depth):.0f} m")

### 2b. Which edges are open?

Expected: **south** (NW Caribbean → Yucatán inflow) and **east** (Florida Straits outflow).
**north** and **west** should be all land. If not, adjust the domain before continuing.

In [ ]:
import numpy as np

wet = np.asarray(topo.tmask) > 0          # 1 = ocean
assert wet.shape == (grid.ny, grid.nx), wet.shape

edges = {"south": wet[0, :], "north": wet[-1, :], "west": wet[:, 0], "east": wet[:, -1]}
for name, e in edges.items():
    print(f"{name:5s}: {int(e.sum()):4d} wet of {e.size}")

BOUNDARIES = [name for name, e in edges.items() if e.any()]
print("open boundaries:", BOUNDARIES)
if {"north", "west"} & set(BOUNDARIES):
    print("WARNING: north/west edge has water -- review the domain before going on.")

## 3. Vertical grid

In [ ]:
from CrocoDash.vgrid import VGrid

vgrid = VGrid.hyperbolic(nk=NK, depth=topo.max_depth, ratio=50.0)
print("top layer thickness ~", float(vgrid.zi[1] - vgrid.zi[0]), "m")

## 4. CESM case + forcings

`CR_JRA` = ocean-only MOM6 with JRA55-do data atmosphere. Sea-level pressure is passed to MOM6, so the inverse-barometer effect is in η (report §A2).

In [ ]:
from CrocoDash.case import Case

case = Case(
    cesmroot=CESMROOT,
    caseroot=CASEROOT,
    inputdir=INPUTDIR,
    ocn_grid=grid,
    ocn_vgrid=vgrid,
    ocn_topo=topo,
    compset="CR_JRA",
    machine="derecho",
    project=PROJECT,
    ntasks_ocn=NTASKS_OCN,
    job_queue=JOB_QUEUE,
    job_wallclock_time=WALLCLOCK,
    override=True,
)

In [ ]:
# Sets RUN_STARTDATE / STOP_N from DATE_RANGE and aligns JRA stream years to the event year.
case.configure_forcings(
    date_range=DATE_RANGE,
    boundaries=BOUNDARIES,
    product_name="GLORYS",
    function_name="get_glorys_data_from_rda",
    tpxo_elevation_filepath=TPXO_H,
    tpxo_velocity_filepath=TPXO_U,
    tidal_constituents=TIDAL_CONSTITUENTS,
)

In [ ]:
# Slowest step: cuts GLORYS IC + OBCs from RDA, regrids them, and regrids TPXO onto each open edge.
case.process_forcings()

### 4b. Extra MOM6 parameters

- CrocoDash turns on only `TIDE_M2` for the body (astronomical) tide (`CrocoDash/forcing/tides.py:86`). Galveston is diurnal, so turn on the rest.
- CrocoDash sets no `DT` for custom grids. 600 s matches the SeaSloth benchmark runs at ~0.1°.

Entries outside CrocoDash's own `! >>> CrocoDash` blocks are kept and take precedence (`Case.configure_forcings` docstring).

In [ ]:
import re

TAG = "! >>> added by 01_build_gulf_mom6.ipynb"
wanted = {"DT": "600.0"}
wanted.update({f"TIDE_{c}": "True" for c in TIDAL_CONSTITUENTS if c != "M2"})

user_nl = CASEROOT / "user_nl_mom"
text = user_nl.read_text()

# Parameters already set anywhere in user_nl_mom (MOM_interface rejects duplicates).
present = set()
for line in text.splitlines():
    s = line.strip()
    if not s or s.startswith("!"):
        continue
    s = s.replace("#override", "").strip()
    m = re.match(r"(\w+)\s*=", s)
    if m:
        present.add(m.group(1))

missing = {k: v for k, v in wanted.items() if k not in present}
if missing:
    block = [TAG] + [f"{k} = {v}" for k, v in missing.items()] + ["! <<< added by 01_build_gulf_mom6.ipynb", ""]
    user_nl.write_text(text.rstrip() + "\n\n" + "\n".join(block))
print("added:", list(missing) or "nothing (all already present)")
print("skipped (already set):", sorted(set(wanted) & present))


## 5. Output for SFINCS (diag_table)

Default output is daily-mean `zos`. That is the **wrong** variable for surge: it adds p_atm/(ρg) and removes the domain mean (report §3).
This step adds:

| file | field | frequency | region |
|---|---|---|---|
| `sfincs_TX` | `SSH_inst` (instantaneous) | 10 min | Texas coast, 97.5–93°W, 27.5–30.0°N (Galveston SFINCS + gauges) |
| `sfincs_MS` | `SSH_inst` | 10 min | LA/MS/AL coast, 91–87.5°W, 28.5–30.8°N (Katrina gauges) |
| `ssh_hourly` | `SSH_inst` (instantaneous) | 1 h | whole domain (maps, sanity checks) |

`regional_section` longitudes use the grid's **0–360** convention.

**No `./preview_namelists` needed first.** The cell writes a complete `SourceMods/src.mom/diag_table`:
- if `CaseDocs/diag_table` exists, it keeps CESM's default outputs and adds ours;
- if not, it writes a small complete table (static grid, daily surface fields) plus ours.

It is safe to re-run: the SFINCS block is never added twice.

**Then check on Derecho** (in the case folder):
```bash
./preview_namelists
grep -c "sfincs_TX\|sfincs_MS\|ssh_hourly" $(./xmlquery --value RUNDIR)/diag_table   # must print 6
```

In [ ]:
# Writes SourceMods/src.mom/diag_table. MOM_interface copies a resolved diag_table from
# SourceMods straight into the run folder, so ./preview_namelists is NOT needed first.
import shutil

src = CASEROOT / "CaseDocs" / "diag_table"          # CESM defaults (exists after a preview_namelists)
dst = CASEROOT / "SourceMods" / "src.mom" / "diag_table"
MARK = "### SFINCS nesting output"                  # everything from here on is ours


def box(w, e, s, n):
    # FMS regional_section: lon0 lon1 lat0 lat1 z0 z1, lon in 0-360 like the grid
    return f"{w % 360:.2f} {e % 360:.2f} {s:.2f} {n:.2f} -1 -1"


c = CASENAME
sfincs = [
    "",
    f"{MARK} (added by notebook section 5)",
    f'"{c}.mom6.sfincs_TX", 10, "minutes", 1, "days", "time"',
    f'"{c}.mom6.sfincs_MS", 10, "minutes", 1, "days", "time"',
    f'"{c}.mom6.ssh_hourly", 1, "hours", 1, "days", "time"',
    f'"ocean_model", "SSH_inst", "SSH_inst", "{c}.mom6.sfincs_TX", "all", ".false.", "{box(-97.5, -93.0, 27.5, 30.0)}", 2',
    f'"ocean_model", "SSH_inst", "SSH_inst", "{c}.mom6.sfincs_MS", "all", ".false.", "{box(-91.0, -87.5, 28.5, 30.8)}", 2',
    # SSH_inst (posted every DT), not the SSH mean: the SSH mean is posted once per coupling
    # step and an hourly average of it can be empty -> FMS "write EMPTY buffer" FATAL.
    f'"ocean_model", "SSH_inst", "SSH_inst", "{c}.mom6.ssh_hourly", "all", ".false.", "none", 2',
    "",
]

if src.exists():
    # CESM default outputs, minus any SFINCS block from an earlier run of this cell
    base = src.read_text().split(MARK)[0].rstrip() + "\n"
    how = "CESM default diag_table (CaseDocs) + SFINCS outputs"
else:
    # No CaseDocs yet: write a small, complete diag_table ourselves
    base = "\n".join(
        [f'"MOM6 diagnostic fields table for CESM case: {c}"', "1 1 1 0 0 0", "",
         f'"{c}.mom6.h.static", -1, "days", 1, "days", "time"',
         f'"{c}.mom6.h.sfc_daily", 1, "days", 1, "days", "time"']
        + [f'"ocean_model", "{v}", "{v}", "{c}.mom6.h.static", "all", "none", "none", 2'
           for v in ["areacello", "geolon", "geolat", "deptho", "wet"]]
        + [f'"ocean_model", "{v}", "{v}", "{c}.mom6.h.sfc_daily", "all", ".true.", "none", 2'
           for v in ["tos", "sos", "SSU", "SSV"]]
    ) + "\n"
    how = "minimal diag_table (no CaseDocs yet) + SFINCS outputs"

# MOM6 declares areacello as the AREA measure for deptho. Some minimal or
# previously generated CaseDocs tables omit it, which is fatal at startup.
if '"areacello"' not in base:
    base += f'"ocean_model", "areacello", "areacello", "{c}.mom6.h.static", "all", "none", "none", 2\n'
    how += " + required areacello AREA measure"

dst.parent.mkdir(parents=True, exist_ok=True)
dst.write_text(base + "\n".join(sfincs))
n = sum(1 for line in dst.read_text().splitlines() if any(k in line for k in ("sfincs_TX", "sfincs_MS", "ssh_hourly")))
print(f"wrote {dst}\n  {how}\n  SFINCS lines: {n} (must be 6)")
print("\n".join(sfincs))

## 6. Build and run (Derecho terminal)

```bash
ssh derecho
cd /glade/work/faezehmaghso/crocodile2026/croc_cases/gom12_<event>.001
./preview_namelists                 # re-check: SourceMods diag_table is picked up
qcmd -A UCGD0009 -- ./case.build   # ~20-40 min
./case.submit
qstat -u faezehmaghso
```

**Cost:** 21 days at 1/12° is roughly 25-100 core-hours (report §F). That is tiny.

**While it runs:**
```bash
tail -f /glade/derecho/scratch/faezehmaghso/gom12_<event>.001/run/ocean.stats   # energy, CFL, NaNs
```

**If it crashes:**
- *CFL / velocity truncations near the coast*: raise `MIN_DEPTH` to 9.5, or lower `DT` to 300.
- *"ESMF mesh and MOM6 domain masks are inconsistent"*: a land/sea mismatch at an open edge. Check section 2b.
- Logs are in `run/` or, after archiving, in `$SCRATCH/archive/gom12_<event>.001/logs/*.gz`.

## 7. First checks after the run
1. `ssh_hourly`: tides propagate in from the south/east edges. No blow-up at the corners.
2. `sfincs_TX`: tidal range and phase at Galveston vs NOAA 8771450 (Pier 21), in the **same datum** (report §3; MSL plan).
3. `sfincs_MS`: Katrina signal 28-30 Aug vs Grand Isle / Dauphin Island. Expect it to be **too low** with JRA winds.

Next notebook: `02_mom6_to_sfincs_bzs.ipynb` (extract `SSH_inst` at SFINCS boundary points → datum → `bzs`).

### Submit on the correct queue (workshop `tutorial` reservation)

The **model run** and the **archiver** must use the workshop queue `tutorial`.
CrocoDash creates the case with `main`, so change it **after** `./case.setup --reset` and **before** `./case.submit`:
```bash
./xmlchange JOB_QUEUE=tutorial --force        # --force: CESM doesn't know this workshop queue
./xmlquery JOB_QUEUE,JOB_WALLCLOCK_TIME       # case.run and case.st_archive: tutorial, 02:00:00
./case.submit                                 # the qsub line must show "-q tutorial"
qstat -u faezehmaghso                         # Queue column shows S7606483 = the tutorial reservation
```
- The **build** (`qcmd -A UCGD0009 -- ./case.build`) always runs on the small `develop` queue. That is normal.
- **Already submitted on `main` by mistake?** Cancel both jobs with `qdel <run-id> <archive-id>` (IDs from `qstat -u faezehmaghso`), then run the lines above.
- `qstat -u faezehmaghso` briefly showing nothing right after submitting is normal; check again or run `tail -8 CaseStatus`.
- **After the workshop** the reservation ends: switch back with `./xmlchange JOB_QUEUE=main --force` (and expect longer waits).
